### RAG Application

In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

C:\Users\itpl59\AppData\Local\Temp\ipykernel_17040\338868836.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\itpl59\Desktop\ITPL\Projects\python-examples\RAG_Langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
load_dotenv() 

True

In [3]:
PDF_PATH = "./data/company_data.pdf"

In [4]:
loader = PyPDFLoader(PDF_PATH)
docs = loader.load()  # one Document per page

In [5]:
# 2) Chunk
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
splits = splitter.split_documents(docs)

In [8]:
import os
from google import genai

client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

#print("Available embedding models:")
#for model in client.models.list():
#    if "embedContent" in model.supported_actions:
#        print(model.name)
#Available embedding models:
#models/gemini-embedding-001
#models/gemini-embedding-2-preview
#models/gemini-embedding-2

# 3) Embed + index
# emb = OpenAIEmbeddings(model="text-embedding-3-small")
emb = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")
#emb = GoogleGenerativeAIEmbeddings(
#    model="models/text-embedding-004",
#    # Forces connection directly through Google AI Studio endpoint
#    client_options={"api_endpoint": "generativelanguage.googleapis.com"}
#)
vs = FAISS.from_documents(splits, emb)
retriever = vs.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [9]:
# 4) Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer ONLY from the provided context. If not found, say you don't know."),
    ("human", "Question: {question}\n\nContext:\n{context}")
])

In [10]:
# 5) Chain
#llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash", # You can also use the available models from your check
    temperature=0
)
def format_docs(docs): return "\n\n".join(d.page_content for d in docs)

parallel = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

chain = parallel | prompt | llm | StrOutputParser()

In [13]:
# 6) Ask questions
print("PDF RAG ready. Ask a question (or Ctrl+C to exit).")
q = input("\nQ: ")
ans = chain.invoke(q.strip())
print("\nA:", ans)

PDF RAG ready. Ask a question (or Ctrl+C to exit).


c:\Users\itpl59\Desktop\ITPL\Projects\python-examples\RAG_Langchain\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



A: I don't know.
